In [0]:
# Gold — Inspect Product Dimension Sources

crm_products = spark.table("bike_lakehouse.silver.crm_products")
erp_categories = spark.table("bike_lakehouse.silver.erp_product_categories")

print("CRM Products:", crm_products.count())
print("ERP Categories:", erp_categories.count())

display(crm_products.limit(10))
display(erp_categories.limit(10))

In [0]:
# Gold — Inspect Product Key Structure

display(
    crm_products
    .select("product_id", "product_key", "product_name", "product_line")
    .orderBy("product_id")
    .limit(30)
)

In [0]:
# Gold — Inspect ERP Product Categories

display(
    erp_categories
    .orderBy("category_id")
)

In [0]:
# Gold — Validate Product to Category Mapping

from pyspark.sql.functions import split, concat_ws

products_with_category = (
    crm_products
    .withColumn(
        "category_id",
        concat_ws(
            "_",
            split("product_key", "-").getItem(0),
            split("product_key", "-").getItem(1)
        )
    )
)

product_category_check = (
    products_with_category
    .join(
        erp_categories,
        products_with_category.category_id == erp_categories.category_id,
        "left"
    )
)

print(
    "Products without category match:",
    product_category_check
    .filter(erp_categories.category_id.isNull())
    .count()
)

display(
    product_category_check
    .select(
        products_with_category.product_key,
        products_with_category.category_id,
        erp_categories.category,
        erp_categories.subcategory,
        erp_categories.maintenance_required
    )
    .limit(20)
)

In [0]:
# Gold — Investigate Unmatched Products

display(
    product_category_check
    .filter(erp_categories.category_id.isNull())
    .select(
        products_with_category.product_id,
        products_with_category.product_key,
        products_with_category.product_name,
        products_with_category.product_line,
        products_with_category.category_id
    )
    .orderBy("product_id")
)

In [0]:
# Gold — Validate CO_PE → CO_PD Mapping

display(
    crm_products
    .filter(crm_products.product_key.startswith("CO-PE-"))
    .select(
        "product_id",
        "product_key",
        "product_name",
        "product_line"
    )
    .orderBy("product_id")
)

display(
    erp_categories
    .filter(erp_categories.category_id == "CO_PD")
)

In [0]:
# Gold — Apply Cross-System Category Mapping

from pyspark.sql.functions import when

products_with_category = products_with_category.withColumn(
    "category_id",
    when(
        products_with_category.category_id == "CO_PE",
        "CO_PD"
    ).otherwise(products_with_category.category_id)
)

product_category_check = (
    products_with_category
    .join(
        erp_categories,
        products_with_category.category_id == erp_categories.category_id,
        "left"
    )
)

print(
    "Products without category match:",
    product_category_check
    .filter(erp_categories.category_id.isNull())
    .count()
)

In [0]:
# Gold — Build Product Dimension

df_dim_products = (
    product_category_check
    .select(
        products_with_category.product_id,
        products_with_category.product_key,
        products_with_category.product_name,
        products_with_category.product_cost,
        products_with_category.product_line,
        products_with_category.start_date,
        products_with_category.end_date,
        erp_categories.category,
        erp_categories.subcategory,
        erp_categories.maintenance_required
    )
)

display(
    df_dim_products
    .orderBy("product_id")
    .limit(20)
)

In [0]:
# Gold — Validate Product Dimension

print("Row count:", df_dim_products.count())

print(
    "Duplicate product IDs:",
    df_dim_products.groupBy("product_id")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "NULL product IDs:",
    df_dim_products.filter("product_id IS NULL").count()
)

print(
    "NULL product keys:",
    df_dim_products.filter("product_key IS NULL").count()
)

print(
    "NULL product names:",
    df_dim_products.filter("product_name IS NULL").count()
)

print(
    "NULL categories:",
    df_dim_products.filter("category IS NULL").count()
)

print(
    "NULL subcategories:",
    df_dim_products.filter("subcategory IS NULL").count()
)

In [0]:
# Gold — Write Product Dimension

df_dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.gold.dim_products")

In [0]:
# Gold — Verify Product Dimension

df_check = spark.table("bike_lakehouse.gold.dim_products")

print("Gold product row count:", df_check.count())

display(
    df_check
    .orderBy("product_id")
    .limit(10)
)